### Exploração de preditores para o modelo de baixada de empresas

**Entrada:** `data/dataset_modelagem_baixada.parquet` (46.454 linhas), com o alvo `alvo_baixada` já definido (ATIVA vs. BAIXADA por motivo de fracasso real).

**Objetivo:** mapear a qualidade das colunas e testar, de forma exploratória, quais variáveis parecem associadas a `alvo_baixada` antes de decidir o que entra no modelo.

#### 1. Carregamento dos dados e configuração de exibição

Ajusta a exibição do pandas e carrega a base de estabelecimentos com o alvo `alvo_baixada` já definido.

In [1]:
# pandas para manipulação tabular; scipy.stats para os testes de associação (Mann-Whitney e qui-quadrado) usados mais adiante
import pandas as pd
from scipy import stats

In [2]:
# evita truncamento ao inspecionar um dataframe com 44 colunas e valores de texto longos (ex.: top3 de categorias)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)

# base de estabelecimentos já com o alvo de baixada definido, uma linha por CNPJ
df = pd.read_parquet('../data/dataset_modelagem_baixada.parquet')
print(df.shape)
df.head()

(46454, 44)


,cnpj_basico,cnpj_ordem,cnpj_dv,identificador_matriz_filial,nome_fantasia,situacao_cadastral,data_situacao_cadastral,motivo_situacao_cadastral,nome_cidade_exterior,pais,...,capital_social,porte_empresa,ente_federativo_responsavel,optante_simples,data_opcao_simples,data_exclusao_simples,optante_mei,data_opcao_mei,data_exclusao_mei,alvo_baixada
0,33615059,0001,43,1,STILOMOR,02,20190514,00,NaN,NaN,...,30000.0,01,NaN,True,20190514,00000000,False,20190514,20210531,0
1,27907876,0001,72,1,CASA PORCELANATO,02,20170606,00,NaN,NaN,...,110000.0,01,NaN,True,20170606,00000000,False,00000000,00000000,0
2,31755936,0001,56,1,NaN,02,20181014,00,NaN,NaN,...,5000.0,01,NaN,False,20181014,20241231,False,20181014,20241231,0
3,33006661,0001,83,1,NaN,02,20190312,00,NaN,NaN,...,500.0,01,NaN,False,20190312,20251231,False,20190312,20251231,0
4,28310042,0001,48,1,NaN,02,20170728,00,NaN,NaN,...,10000.0,01,NaN,True,20170728,00000000,False,00000000,00000000,0


#### 2. Perfil geral das colunas

Monta um resumo coluna a coluna (tipo, nulos, cardinalidade, valores mais comuns) para decidir quais colunas têm qualidade suficiente para virar preditor.

In [3]:
perfil = []
for col in df.columns:
    dtype = df[col].dtype
    pct_nulos = df[col].isna().mean() * 100
    n_unicos = df[col].nunique(dropna=True)
    if pd.api.types.is_numeric_dtype(df[col]):
        # numéricas: resume pela distribuição (min/mediana/max)
        resumo = f"min={df[col].min()}, mediana={df[col].median()}, max={df[col].max()}"
    else:
        # categóricas/texto: resume pelas categorias mais frequentes
        top3 = df[col].value_counts(dropna=True).head(3).to_dict()
        resumo = f"top3={top3}"
    perfil.append({
        'coluna': col,
        'dtype': str(dtype),
        'pct_nulos': round(pct_nulos, 2),
        'n_unicos': n_unicos,
        'resumo': resumo
    })

perfil_df = pd.DataFrame(perfil)
# persiste o perfil em disco para consulta rápida sem precisar re-rodar o notebook inteiro
perfil_df.to_csv('../data/perfil_colunas_dataset_modelagem.csv', index=False)
perfil_df

,coluna,dtype,pct_nulos,n_unicos,resumo
0,cnpj_basico,str,0.00,46186,"top3={'61585865': 17, '47960950': 15, '88212113': 10}"
1,cnpj_ordem,str,0.00,282,"top3={'0001': 44066, '0002': 1123, '0003': 324}"
2,cnpj_dv,str,0.00,100,"top3={'00': 1484, '40': 821, '05': 800}"
3,identificador_matriz_filial,str,0.00,2,"top3={'1': 44070, '2': 2384}"
4,nome_fantasia,str,45.77,24228,"top3={'COMERCIO': 17, 'CACAU SHOW': 12, 'O BOTICARIO': 11}"
5,situacao_cadastral,str,0.00,2,"top3={'08': 27656, '02': 18798}"
6,data_situacao_cadastral,str,0.00,3273,"top3={'20190531': 66, '20190514': 63, '20190320': 63}"
7,motivo_situacao_cadastral,str,0.00,6,"top3={'01': 27635, '00': 18798, '05': 18}"
8,nome_cidade_exterior,str,100.00,2,"top3={'LONDON': 1, 'LEWES': 1}"
9,pais,str,100.00,2,"top3={'628': 1, '249': 1}"


#### 3. Construção da idade da empresa

Deriva a idade da empresa (dias e anos) a partir de `data_inicio_atividade` e da data de referência, para poder testá-la como preditor numérico.

In [4]:
# data de referência da extração dos dados (conforme o doc do projeto: 08/08/2026)
data_referencia = pd.Timestamp('2026-08-08')

# converte a string AAAAMMDD em data real; datas inválidas viram NaT em vez de quebrar a conversão
df['data_inicio_atividade_dt'] = pd.to_datetime(df['data_inicio_atividade'], format='%Y%m%d', errors='coerce')
# idade da empresa até a data de referência: tende a ser um preditor relevante de sobrevivência/fracasso
df['idade_empresa_dias'] = (data_referencia - df['data_inicio_atividade_dt']).dt.days
df['idade_empresa_anos'] = df['idade_empresa_dias'] / 365.25

df[['data_inicio_atividade', 'idade_empresa_dias', 'idade_empresa_anos']].describe()

,idade_empresa_dias,idade_empresa_anos
count,46454.000000,46454.000000
mean,2936.907543,8.040815
std,313.592233,0.858569
min,2412.000000,6.603696
25%,2663.000000,7.290897
50%,2923.000000,8.002738
75%,3210.000000,8.788501
max,3506.000000,9.598905


#### 4. Teste de associação -- variáveis numéricas

Compara `idade_empresa_anos` e `capital_social` entre ativas e fracassadas com Mann-Whitney, adequado por serem variáveis assimétricas.

In [5]:
# preditores numéricos candidatos a testar contra o alvo
candidatos_numericos = ['idade_empresa_anos', 'capital_social']

print("=== VARIÁVEIS NUMÉRICAS (Mann-Whitney) ===\n")
for col in candidatos_numericos:
    grupo0 = df.loc[df['alvo_baixada'] == 0, col].dropna()
    grupo1 = df.loc[df['alvo_baixada'] == 1, col].dropna()
    # Mann-Whitney: compara as distribuições entre os dois grupos sem supor normalidade,
    # adequado aqui porque idade e capital_social são bem assimétricas
    stat, p = stats.mannwhitneyu(grupo0, grupo1, alternative='two-sided')
    print(f"{col}:")
    print(f"  mediana ATIVA (0)    = {grupo0.median():.2f}")
    print(f"  mediana FRACASSO (1) = {grupo1.median():.2f}")
    print(f"  p-valor              = {p:.2e}")
    print()

=== VARIÁVEIS NUMÉRICAS (Mann-Whitney) ===

idade_empresa_anos:
  mediana ATIVA (0)    = 7.89
  mediana FRACASSO (1) = 8.08
  p-valor              = 3.49e-62

capital_social:
  mediana ATIVA (0)    = 10000.00
  mediana FRACASSO (1) = 5000.00
  p-valor              = 0.00e+00



#### 5. Teste de associação -- variáveis categóricas

Aplica qui-quadrado às variáveis categóricas candidatas e complementa com a taxa de fracasso por categoria, já que o teste não indica a direção do efeito.

In [6]:
# preditores categóricos candidatos a testar contra o alvo
candidatos_categoricos = ['identificador_matriz_filial', 'porte_empresa', 'optante_simples',
                           'optante_mei', 'natureza_juridica', 'qualificacao_responsavel',
                           'uf', 'cnae_fiscal_principal']

print("=== VARIÁVEIS CATEGÓRICAS (qui-quadrado) ===\n")
for col in candidatos_categoricos:
    # qui-quadrado: testa se a distribuição de alvo_baixada muda entre as categorias da variável
    tabela = pd.crosstab(df[col], df['alvo_baixada'])
    chi2, p, dof, esperado = stats.chi2_contingency(tabela)
    n_categorias = df[col].nunique()
    print(f"{col} ({n_categorias} categorias) — qui-quadrado={chi2:.1f}, p={p:.2e}, gl={dof}")

    # taxa de fracasso por categoria (só as 10 mais populosas), para saber a direção do efeito,
    # já que o teste só diz que existe associação, não qual categoria puxa o risco pra cima ou pra baixo
    taxa = (df.groupby(col)['alvo_baixada']
              .agg(n='count', taxa_fracasso='mean')
              .sort_values('n', ascending=False)
              .head(10))
    print(taxa)
    print()

=== VARIÁVEIS CATEGÓRICAS (qui-quadrado) ===

identificador_matriz_filial (2 categorias) — qui-quadrado=18.3, p=1.91e-05, gl=1
                                 n  taxa_fracasso
identificador_matriz_filial                      
1                            44070       0.597617
2                             2384       0.553272

porte_empresa (3 categorias) — qui-quadrado=645.5, p=6.79e-141, gl=2
                   n  taxa_fracasso
porte_empresa                      
01             43519       0.610170
03              1885       0.403183
05              1050       0.325714

optante_simples (2 categorias) — qui-quadrado=29289.8, p=0.00e+00, gl=1
                     n  taxa_fracasso
optante_simples                      
False            31647       0.861946
True             14807       0.025528

optante_mei (2 categorias) — qui-quadrado=15728.5, p=0.00e+00, gl=1
                 n  taxa_fracasso
optante_mei                      
False        37751       0.732484
True          8703       0.

uf (28 categorias) — qui-quadrado=148.4, p=1.00e-18, gl=27
        n  taxa_fracasso
uf                      
SP  11349       0.592651
MG   5161       0.621197
RJ   3604       0.582408
BA   2978       0.582606
PR   2977       0.576419
RS   2718       0.611847
SC   1950       0.614872
CE   1839       0.613921
GO   1826       0.617744
PE   1676       0.628878



cnae_fiscal_principal (76 categorias) — qui-quadrado=1039.4, p=1.27e-169, gl=75
                           n  taxa_fracasso
cnae_fiscal_principal                      
4781400                11433       0.656521
4712100                 3907       0.570770
4723700                 2600       0.652308
4772500                 2520       0.625794
4729699                 1955       0.631202
4744099                 1444       0.452909
4789099                 1424       0.601124
4789004                 1164       0.594502
4724500                 1061       0.589067
4751201                  984       0.639228



#### 6. Checagem de consistência: baixa vs. exclusão do Simples

Confere, entre as fracassadas fora do Simples, se a exclusão do Simples e a mudança de situação cadastral são o mesmo evento administrativo.

In [7]:
# verifica se a data de exclusão do Simples coincide com a data da mudança de situação cadastral --
# se coincidir na maioria dos casos, os dois campos registram o mesmo evento administrativo,
# e não dois sinais independentes de que a empresa fracassou
df['data_situacao_cadastral_dt'] = pd.to_datetime(df['data_situacao_cadastral'], format='%Y%m%d', errors='coerce')
df['data_exclusao_simples_dt'] = pd.to_datetime(df['data_exclusao_simples'], format='%Y%m%d', errors='coerce')

fracasso_nao_simples = df[(df['alvo_baixada']==1) & (df['optante_simples']==False)]
print("Fracassadas com optante_simples=False:", len(fracasso_nao_simples))
print("Dessas, com data_exclusao_simples preenchida (≠ 00000000):",
      fracasso_nao_simples['data_exclusao_simples_dt'].notna().sum())

diff_dias = (fracasso_nao_simples['data_situacao_cadastral_dt'] -
             fracasso_nao_simples['data_exclusao_simples_dt']).dt.days
print(diff_dias.describe())

Fracassadas com optante_simples=False: 27278
Dessas, com data_exclusao_simples preenchida (≠ 00000000): 26571
count    26571.000000
mean        30.482067
std        284.165379
min      -3070.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       6819.000000
dtype: float64


#### 7. Cruzamento natureza_juridica × qualificacao_responsavel

Cruza as duas variáveis para entender sua relação de dependência antes de usá-las como preditores separados.

In [8]:
# cruza as duas variáveis para checar dependência entre elas; é aqui que fica visível a associação
# quase 1:1 entre natureza_juridica=2305 (EIRELI) e qualificacao_responsavel=65 (578 vs. 575 casos) --
# pista de que 2305 é um artefato de reclassificação legal, investigado a fundo no notebook 02
pd.crosstab(df['natureza_juridica'], df['qualificacao_responsavel'])

qualificacao_responsavel,05,10,12,16,17,49,50,64,65
natureza_juridica,,,,,,,,,
2038,0,0,0,1,0,0,0,0,0
2046,0,73,0,22,0,0,0,3,0
2054,2,45,0,44,0,0,0,8,0
2062,299,0,4,0,0,8107,0,33,0
2127,1,0,0,0,0,0,0,0,0
2135,0,0,2,0,0,0,37212,0,0
2143,0,0,0,11,0,0,0,0,0
2216,0,0,0,0,2,0,0,0,0
2240,0,0,0,0,0,5,0,0,0
